# Chapter 25: Unit Tests

Chapter 25 is the chapter this series has been quietly rehearsing for: every notebook
so far ended its sections in **check blocks** — assertions that decide, mechanically,
whether an exercise passed. That machinery has a name in Rust, and it is the same
machinery Python programmers know as **`unittest`**: write a function whose only job
is to *fail loudly when the code is wrong*, and let a runner discover, call, and
count those functions. Rust's version is the `#[test]` attribute plus the `assert`
family, driven by **libtest** — the harness that `cargo test` runs and the same
runner you meet raw here.

The chapter's honest lane is forced by the kernel: **evcxr strips `#[test]`
functions** (they are `cfg(test)`-gated, and the kernel builds without the test
harness — probe receipt: calling one from the *next cell* is `E0425`). So the
notebook does what chapter 23 did for `docker`: it **builds its own test desk**.
§0 writes small test suites into `scratch25/`, compiles them with `rustc --test`,
and runs the harness binary — so every receipt in this chapter is the *real*
libtest output, captured as data: `running 6 tests`, `- should panic`,
`ignored, slow stand-in`, `test result: FAILED. 1 passed; 1 failed`, exit `101`.

The running domain stays the trading desk's: a `parse_fill` family small enough to
test exhaustively. The Python mapping is explicit throughout, and §9 puts the whole
`unittest` → Rust dictionary on one table.


## Contents

- [0. Notebook Conventions: Tests Are Functions](#0-notebook-conventions-tests-are-functions)
- [1. Your First Test](#1-your-first-test)
- [2. The Assert Family](#2-the-assert-family)
- [3. Expected Panics](#3-expected-panics)
- [4. Result-Returning Tests](#4-result-returning-tests)
- [5. Choosing What Runs](#5-choosing-what-runs)
- [6. Fixtures: Setup and Teardown](#6-fixtures-setup-and-teardown)
- [7. Organizing Tests](#7-organizing-tests)
- [8. Exit Codes and the Result Line](#8-exit-codes-and-the-result-line)
- [9. The unittest Mapping](#9-the-unittest-mapping)
- [10. E-Codes, Traps, and Kernel Facts](#10-e-codes-traps-and-kernel-facts)
- [11. Mental Model: The Receipts](#11-mental-model-the-receipts)
- [12. Cheat Sheet](#12-cheat-sheet)
- [13. Review and What Comes Next](#13-review-and-what-comes-next)


## 0. Notebook Conventions: Tests Are Functions

The standing rules all hold: no `fn main` in cells, state persists across cells,
items are order-independent, one namespace per notebook, and the first cell that
needs a name does the `use`. Chapter 25 is **std-only and fully synchronous** — no
`:dep`, no runtimes, the lightest chapter since 13. Four chapter-specific
conventions:

- **The kernel has no test runner.** evcxr compiles each cell *without* the test
  harness, and `#[test]` is `cfg(test)`-gated — so a `#[test]` function is stripped
  from the cell it is written in, even before persistence. The probe tried to call
  `k_ok()` in the very next cell: `E0425 cannot find function`. `cfg!(test)` is
  `false` (§7.3). **In this notebook `#[test]` appears only inside desk sources** —
  string literals the desk writes to disk for a real `rustc --test` build.
- **The desk is libtest itself.** `rustc --test file.rs` compiles with
  `cfg(test)` *on* and links the harness — the same runner `cargo test` drives.
  Output lines here (`running 6 tests`, `test result: ok. 5 passed; ...`) are
  exactly what `cargo test` prints, minus cargo's own lines. A desk build lands in
  ~0.9 s on this toolchain; a suite runs in ~10 ms.
- **Failure output is data, never a dying cell.** A gated notebook cannot ship a
  cell that panics, so every failing-test receipt arrives *captured*: the desk runs
  the harness as a child process and returns its stdout and exit code as strings —
  chapter 23's subprocess discipline feeding chapter 25's.
- **`scratch25/` is the desk directory.** Sources land as `desk_<name>.rs`, binaries
  as `desk_<name>.exe`; §13.1 removes the directory when the chapter is done.


In [ ]:
// 0.1 Smoke test: the kernel can reach the toolchain -- and `--test` is a
// plain rustc flag, not a cargo feature.
use std::process::Command;

let out = Command::new("rustc").arg("--version").output().unwrap();
println!("toolchain: {}", String::from_utf8_lossy(&out.stdout).trim());


In [ ]:
// 0.2 The two desk helpers every later cell runs on.
use std::fs;

/// Write `src` to `scratch25/desk_<name>.rs` and compile it, with the test
/// harness on (`test_mode = true`, i.e. `rustc --test`) or off. Returns the
/// compiler's stderr as a String -- compile errors and warnings come back as
/// data, so a cell that *demonstrates* a compile failure never fails itself.
fn desk_stderr(name: &str, src: &str, test_mode: bool) -> String {
    fs::create_dir_all("scratch25").unwrap();
    let rs = format!("scratch25/desk_{name}.rs");
    let exe = format!("scratch25/desk_{name}.exe");
    fs::write(&rs, src).unwrap();
    let mut cmd = Command::new("rustc");
    cmd.args(["--edition", "2024"]);
    if test_mode {
        cmd.arg("--test");
    }
    let out = cmd.args([&rs, "-o", &exe]).output().unwrap();
    String::from_utf8_lossy(&out.stderr).into_owned()
}

/// Compile `src` with the harness on and run the test binary with `args`.
/// Returns `(stdout, exit_code)`. A desk source that does not compile is a
/// bug in this notebook, so the compile failure panics with its message;
/// a *test* that fails at runtime is data, like any other receipt.
fn desk_run(name: &str, src: &str, args: &[&str]) -> (String, i32) {
    let stderr = desk_stderr(name, src, true);
    assert!(
        stderr.is_empty(),
        "desk_{name} failed to compile:\n{stderr}"
    );
    let run = Command::new(format!("./scratch25/desk_{name}.exe"))
        .args(args)
        .output()
        .unwrap();
    let code = run.status.code().unwrap_or(-1);
    (String::from_utf8_lossy(&run.stdout).into_owned(), code)
}

println!("desk helpers ready (desk_stderr, desk_run)");


## 1. Your First Test

Python's smallest test is a `unittest.TestCase` method: a class, a method named
`test_*`, a runner that finds it. Rust has no class and no runner to write — a
**test is a plain function marked `#[test]`**, and libtest finds them the way
`unittest.main()` finds methods. Two moves to internalize:

- the function takes **no arguments** and returns either nothing or a `Result`
  (§4) — the harness calls it and counts a panic or an `Err` as a failure;
- the harness *is a binary*: `rustc --test` builds one, and running it executes
  every `#[test]` function it contains, printing one line per test.

First the idea in kernel-honest form — a test is just a function you call — then
the real thing on the desk.


In [ ]:
// 1.1 Tests are functions: the kernel can run this idea, minus the attribute.
fn normalize_sym(raw: &str) -> String {
    raw.trim().to_uppercase()
}

// The body of a test is nothing special: arrange, act, assert.
let given = " aapl ";
let got = normalize_sym(given);
assert_eq!(got, "AAPL");
println!("a test, run by hand: normalize_sym({given:?}) = {got:?}");


In [ ]:
// 1.2 The real thing: a two-test suite, built and run by the harness.
const FIRST_SUITE: &str = r#"fn normalize_sym(raw: &str) -> String {
    raw.trim().to_uppercase()
}

#[test]
fn keeps_short_symbols() {
    assert_eq!(normalize_sym("t"), "T");
}

#[test]
fn normalizes_case() {
    assert_eq!(normalize_sym(" aapl "), "AAPL");
}
"#;

let (out, code) = desk_run("first", FIRST_SUITE, &[]);
println!("{out}");
println!("exit code: {code}");
assert_eq!(code, 0);
assert!(out.contains("running 2 tests"));
assert!(out.contains("test result: ok. 2 passed; 0 failed"));


Read the receipt like a harness would:

- `running 2 tests` — discovery found both `#[test]` functions; the desk did not
  call anything by hand.
- the lines print in **alphabetical order** of the test's full path —
  `keeps_short_symbols` before `normalizes_case` regardless of source order
  (execution itself is parallel across threads; the *report* is sorted).
- `test result: ok. 2 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out` —
  the counts line every CI system greps for; §8 parses it as data.
- `finished in 0.00s` — the suite's wall time.

One test failing changes the *whole binary's* verdict, not just its line:


In [ ]:
// 1.3 One failing test: the suite's verdict flips, and the exit code says so.
const FIRST_BROKEN: &str = r#"fn normalize_sym(raw: &str) -> String {
    raw.trim().to_uppercase()
}

#[test]
fn normalizes_case() {
    assert_eq!(normalize_sym(" aapl "), "AAPL");
}

#[test]
fn trims_spaces() {
    assert_eq!(normalize_sym(" msft "), "MSFT ");
}
"#;

let (out, code) = desk_run("broken", FIRST_BROKEN, &[]);
println!("{out}");
println!("exit code: {code}");
assert_eq!(code, 101);
assert!(out.contains("test trims_spaces ... FAILED"));
assert!(out.contains("test result: FAILED. 1 passed; 1 failed"));


The two conventions to take from this receipt: `101` is libtest's **any-test-failed
exit code** (the CI contract — `0` green, `101` red), and the failure block prints
the failing test's *captured output* under `---- trims_spaces stdout ----` — the
panic message with the source location, plus the same `failures:` list a second
time at the bottom. §2 zooms into that block, because the assert macros write it.


## 2. The Assert Family

Chapter 08 met the `assert` macros as *runtime checks*; here they are the **test
vocabulary**, the direct map of `unittest`'s assertion methods: `assert_eq!` for
`assertEqual`, `assert_ne!` for `assertNotEqual`, `assert!` for `assertTrue` — and
`matches!` inside `assert!` doing the work of `assertIsInstance`-style shape checks
(`Option`/`Result`/enum, chapter 08's pattern matching as an assertion).

What makes them *test* macros is the failure payload: each one panics with a
message that names the macro, carries the custom message if you gave one, and
prints the compared values — which is exactly the block libtest captures per
failing test. The desk captures all four shapes in one run:


In [ ]:
// 2.1 assert_eq! failure anatomy: the message line, then the values.
const ASSERT_EQ_FAIL: &str = r#"fn qty_of(line: &str) -> i64 {
    line.split(',').nth(1).unwrap().trim().parse().unwrap()
}

#[test]
fn parses_qty() {
    assert_eq!(qty_of("AAPL,100"), 100, "qty mismatch for {}", "AAPL");
}

#[test]
fn wrong_qty() {
    assert_eq!(qty_of("MSFT,50"), 60, "qty mismatch for {}", "MSFT");
}
"#;

let (out, code) = desk_run("eqfail", ASSERT_EQ_FAIL, &[]);
let section = out
    .split("---- wrong_qty stdout ----")
    .nth(1)
    .unwrap()
    .split("\n\nfailures:")
    .next()
    .unwrap()
    .to_string();
println!("captured section:\n{section}");
assert!(section.contains("assertion `left == right` failed: qty mismatch for MSFT"));
assert!(section.contains(" left: 50"));
assert!(section.contains(" right: 60"));
assert_eq!(code, 101);


The anatomy, line by line:

- `assertion \`left == right\` failed: qty mismatch for MSFT` — macro name in
  backticks, then the **custom message** (the extra `assert_eq!` format arguments,
  same trailing-args rule as `format!`).
- `left: 50` / `right: 60` — the two expressions' values, `Debug`-formatted
  (that is why the compared types must implement `Debug` — chapter 08's E0369
  receipt, back as a requirement).
- `note: run with \`RUST_BACKTRACE=1\` ...` — a backtrace is available but a
  failing assert is usually self-explanatory.

`assert!`, `assert_ne!` and the `matches!` idiom print their own shapes — note that
bare `assert!` without a message prints the *expression text*:


In [ ]:
// 2.2 The other three shapes: assert! + message, assert_ne!, matches! in assert!.
const ASSERT_SHAPES: &str = r#"#[test]
fn never_negative() {
    let qty: i64 = -5;
    assert!(qty >= 0, "qty must be non-negative, got {qty}");
}

#[test]
fn symbols_differ() {
    assert_ne!("AAPL", "AAPL");
}

#[test]
fn bare_assert() {
    let ok = false;
    assert!(ok);
}

#[test]
fn shape_is_fill() {
    let v: Result<i64, String> = Err("bad".to_string());
    assert!(matches!(v, Ok(_)));
}
"#;

let (out, code) = desk_run("shapes", ASSERT_SHAPES, &[]);
for probe in [
    "qty must be non-negative, got -5",
    "assertion `left != right` failed",
    " left: \"AAPL\"",
    "assertion failed: ok",
    "assertion failed: matches!(v, Ok(_))",
] {
    assert!(out.contains(probe), "missing shape {probe:?} in:\n{out}");
    println!("shape found: {probe:?}");
}
assert_eq!(code, 101);


`subTest`, for the curious: unittest's way of looping several cases inside one test
method while reporting each independently maps in Rust to **separate `#[test]`
functions** — the harness already prints one line and one capture block per test,
so the loop-level granularity unittest fakes with `subTest` is native here. The
`t_asserts` run above is that shape: four independent failures, four sections.

In-kernel, the same payloads arrive through `catch_unwind` (chapter 11): an assert
is still just a panic, and the panic payload is the message:


In [ ]:
// 2.3 The failure payload as data, no harness: catch_unwind + downcast (ch11).
{
    use std::panic::{catch_unwind, AssertUnwindSafe};

    let payload = catch_unwind(AssertUnwindSafe(|| {
        assert_eq!(2 + 2, 5);
    }))
    .unwrap_err();
    let msg = payload
        .downcast_ref::<String>()
        .map(String::as_str)
        .or_else(|| payload.downcast_ref::<&str>().copied())
        .unwrap_or("<no string payload>");
    println!("payload: {msg}");
    assert!(msg.contains("assertion `left == right` failed"));
    assert!(msg.contains("left: 4") && msg.contains("right: 5"));
}


## 3. Expected Panics

`assertRaises` has a direct translation with one twist. Python hands the runner a
context manager and it checks the *exception type*; Rust's `#[should_panic]`
annotates the test itself and checks the **panic message** — the `expected = "…"`
argument is a *substring* match against whatever the test panics with. The harness
marks such tests on their result line:

```
test panics_on_empty - should panic ... ok
```

and if the test *doesn't* panic, that is a failure too — the expectation cuts both
ways. The desk shows the passing shape first, mismatch second:


In [ ]:
// 3.1 #[should_panic(expected = ..)]: the panicking test, passing.
const SHOULD_PANIC_OK: &str = r#"fn parse_qty(s: &str) -> i64 {
    s.trim().parse().expect("qty is not a number")
}

#[test]
#[should_panic(expected = "qty is not a number")]
fn rejects_text_qty() {
    parse_qty("lots");
}
"#;

let (out, code) = desk_run("panicok", SHOULD_PANIC_OK, &[]);
println!("{out}");
assert!(out.contains("test rejects_text_qty - should panic ... ok"));
assert_eq!(code, 0);


The panic message here is `.expect`'s: `qty is not a number: ParseIntError { ... }`
— and the `expected` substring lands inside it, so the match succeeds. Note what
the attribute does **not** promise: it checks neither the exception's type (panics
in Rust are untyped at the boundary — a payload, chapter 11) nor the panic's
location. When the substring does not match, libtest prints a three-line
diff-style note — the most useful failure text in the harness:


In [ ]:
// 3.2 The mismatch: the test panics, but not with the expected substring.
const SHOULD_PANIC_MISS: &str = r#"fn parse_qty(s: &str) -> i64 {
    s.trim().parse().expect("qty is not a number")
}

#[test]
#[should_panic(expected = "negative")]
fn rejects_text_qty() {
    parse_qty("lots");
}
"#;

let (out, code) = desk_run("panicmiss", SHOULD_PANIC_MISS, &[]);
let start = out.find("---- rejects_text_qty stdout ----").unwrap();
let section = out[start..].to_string();
println!("{section}");
assert!(section.contains("note: panic did not contain expected string"));
assert!(section.contains("panic message: \"qty is not a number"));
assert!(section.contains("expected substring: \"negative\""));
assert_eq!(code, 101);


## 4. Result-Returning Tests

A test may return a `Result`: `Ok(())` passes, **any `Err` fails** — the harness
prints the error into the test's capture block and counts the failure, no panic
required. This is the shape for tests whose bodies want `?`: one `parse_qty("…")?`
replaces a nested `match` on setup, and chapter 11's error plumbing works inside a
test exactly as it does in `main`.

The return type is not free: `Result<(), E>` works only because `E: Debug`
(libtest prints it) — the same `Termination` bound `fn main() -> Result` obeys,
§10.2 catches the `i32` that fails it.


In [ ]:
// 4.1 A Result test that passes: `?` on setup, asserts in the middle, Ok(()) last.
const RESULT_OK: &str = r#"fn parse_qty(s: &str) -> Result<i64, std::num::ParseIntError> {
    s.trim().parse::<i64>()
}

#[test]
fn reads_qty() -> Result<(), std::num::ParseIntError> {
    let q = parse_qty(" 100 ")?;
    assert_eq!(q, 100);
    assert!(parse_qty("lots").is_err(), "text must not parse");
    Ok(())
}
"#;

let (out, code) = desk_run("resok", RESULT_OK, &[]);
println!("{out}");
assert!(out.contains("test reads_qty ... ok"));
assert_eq!(code, 0);


In [ ]:
// 4.2 Err means failure: no panic, no assert -- the harness prints the error.
const RESULT_ERR: &str = r#"fn parse_qty(s: &str) -> Result<i64, String> {
    s.trim().parse::<i64>().map_err(|e| e.to_string())
}

#[test]
fn err_means_failure() -> Result<(), String> {
    let q = parse_qty("lots")?;
    assert_eq!(q, 1);
    Ok(())
}
"#;

let (out, code) = desk_run("reserr", RESULT_ERR, &[]);
let start = out.find("---- err_means_failure stdout ----").unwrap();
let section = out[start..].to_string();
println!("{section}");
assert!(section.contains("Error: \"invalid digit found in string\""));
assert_eq!(code, 101);


Read that capture block closely: `Error: "invalid digit found in string"` — the
harness prints the `Err` payload with `Debug`, so the `String` arrives **quoted**.
The same `E: Debug` requirement bit earlier from the other side: in the probe
phase, `parse_fill(..).unwrap_err()` on a non-`Debug` struct refused to compile
with E0277 — `unwrap_err` needs to print the *Ok* value it is throwing away, just
as `unwrap` needs to print the *Err*. `#[should_panic]` and `Result` are the two
doors into "expected failure"; pick per section: panic for contract violations,
`Err` for data-level rejections.


## 5. Choosing What Runs

unittest reaches for decorators and `-k`: `@unittest.skip("reason")` parks a test,
`-k pattern` selects some. libtest bakes all of it into **attributes and command
line arguments**, and this is where the harness binary stops being a formality —
you can run the same suite six ways without recompiling. The rest of the chapter
reuses one six-test suite, so build it once:


In [ ]:
// 5.1 The chapter's standing suite: six tests, one of each interesting shape.
// Later cells re-run THIS source with different arguments -- the desk recompiles
// it (about a second), the arguments change what the harness runs.
const SUITE_SRC: &str = r#"fn parse_fill(s: &str) -> Result<(String, i64), String> {
    if s.is_empty() {
        return Err("empty input".to_string());
    }
    let parts: Vec<&str> = s.split(',').collect();
    if parts.len() != 2 {
        return Err(format!("expected 2 fields, got {}", parts.len()));
    }
    let sym = parts[0].trim().to_uppercase();
    let qty: i64 = parts[1]
        .trim()
        .parse()
        .map_err(|_| format!("bad qty: {}", parts[1]))?;
    Ok((sym, qty))
}

#[cfg(test)]
fn fixture_row() -> String {
    "AAPL, 100".to_string()
}

#[cfg(test)]
mod scan {
    use super::*;

    #[test]
    fn module_path_tests_work() {
        let (sym, _) = parse_fill(&fixture_row()).unwrap();
        assert_eq!(sym, "AAPL");
    }
}

#[test]
fn parses_ok() {
    let (sym, qty) = parse_fill("AAPL, 100").unwrap();
    assert_eq!(sym, "AAPL");
    assert_eq!(qty, 100);
}

#[test]
fn rejects_bad_qty() -> Result<(), String> {
    let (_, qty) = parse_fill("MSFT, 50")?;
    assert_eq!(qty, 50);
    let err = parse_fill("MSFT, lots").unwrap_err();
    assert!(err.contains("bad qty"), "got: {err}");
    Ok(())
}

#[test]
#[should_panic(expected = "empty input")]
fn panics_on_empty() {
    parse_fill("").unwrap();
}

#[test]
#[ignore = "slow stand-in for --ignored"]
fn heavy_demo() {
    std::thread::sleep(std::time::Duration::from_millis(5));
}

#[test]
fn quiet_demo() {
    println!("side effect printed from inside quiet_demo");
    parse_fill("T, 1").unwrap();
}
"#;

let (out, code) = desk_run("suite", SUITE_SRC, &[]);
println!("{out}");
assert_eq!(code, 0);
assert!(out.contains("running 6 tests"));
assert!(out.contains("test heavy_demo ... ignored, slow stand-in for --ignored"));
assert!(out.contains("test panics_on_empty - should panic ... ok"));
assert!(out.contains("test scan::module_path_tests_work ... ok"));
assert!(out.contains("test result: ok. 5 passed; 0 failed; 1 ignored"));


Everything §5 needs is in that one receipt:

- **`#[ignore = "reason"]`** parks a test — and the reason rides the result line:
  `test heavy_demo ... ignored, slow stand-in for --ignored`. It is still counted
  (`1 ignored` in the result line) but not run; `--ignored` inverts the selection
  and runs *only* the parked tests (§5.3).
- **the module path is the test's name.** `scan::module_path_tests_work` is one
  test whose full path includes its `mod` — §7 leans on this.
- `#[should_panic]` tests carry their `- should panic` annotation on the line.

The selection verbs, one per desk run:


In [ ]:
// 5.2 Name filters: a bare argument is a substring match on the full test path.
let (sub, sub_code) = desk_run("suite", SUITE_SRC, &["parse"]);
println!("--- filter 'parse':\n{sub}");
assert!(sub.contains("running 1 test"));
assert!(sub.contains("test parses_ok ... ok"));
assert!(sub.contains("5 filtered out"));
assert_eq!(sub_code, 0);

let (exact, exact_code) = desk_run("suite", SUITE_SRC, &["scan::module_path_tests_work", "--exact"]);
println!("--- exact full path:\n{exact}");
assert!(exact.contains("test scan::module_path_tests_work ... ok"));
assert!(exact.contains("5 filtered out"));
assert_eq!(exact_code, 0);

// A filter that matches nothing is not an error: zero tests ran, verdict ok.
let (none, code) = desk_run("suite", SUITE_SRC, &["nope", "--exact"]);
println!("--- no match:\n{none}");
assert!(none.contains("running 0 tests"));
assert!(none.contains("test result: ok. 0 passed"));
assert!(none.contains("6 filtered out"));
assert_eq!(code, 0);


Two selection facts worth memorizing: the bare filter is a **substring match on
the full path** — `"parse"` caught `parses_ok` but *not* `rejects_bad_qty`, even
though that test's body calls `parse_fill` twice (selection reads names, never
bodies) — and `--exact` takes the **whole path**, which is how you single out a
test inside a `mod`. A zero-match filter exits `0`: "selected nothing, ran
nothing, nothing failed" — the CI quirk §8.2 turns into a warning.

`--list` prints the discovery table without running anything:


In [ ]:
// 5.3 --list: the discovery receipt, one "name: test" line per test.
let (out, code) = desk_run("suite", SUITE_SRC, &["--list"]);
println!("{out}");
assert_eq!(code, 0);
assert!(out.contains("parses_ok: test"));
assert!(out.contains("scan::module_path_tests_work: test"));
assert!(out.contains("6 tests, 0 benchmarks"));


`0 benchmarks` is the harness's other tenant: libtest also ran Rust's benchmark
harness (chapter 21's criterion world is the modern replacement). Then output
visibility — unittest's captured-stdout-vs-`-v` dance has two flags here, and they
disagree pleasantly about *where* the prints land:


In [ ]:
// 5.4 --nocapture streams test prints inline; --show-output archives them
// under a "successes:" section after the run.
let (nc, _) = desk_run("suite", SUITE_SRC, &["quiet_demo", "--nocapture"]);
println!("--- --nocapture:\n{nc}");
let pos = nc.find("running 1 test").unwrap();
let printed = nc.find("side effect printed from inside quiet_demo").unwrap();
let verdict = nc.find("test quiet_demo ... ok").unwrap();
assert!(pos < printed && printed < verdict, "print landed inline");

let (so, _) = desk_run("suite", SUITE_SRC, &["quiet_demo", "--show-output"]);
println!("--- --show-output:\n{so}");
assert!(so.contains("test quiet_demo ... ok"));
assert!(so.contains("successes:"));
assert!(so.contains("---- quiet_demo stdout ----"));
assert!(so.contains("side effect printed from inside quiet_demo"));

{
    let (serial, _) = desk_run("suite", SUITE_SRC, &["--test-threads=1"]);
    let lines: Vec<String> = serial
        .lines()
        .filter(|l| l.starts_with("test "))
        .map(|l| l.to_string())
        .collect();
    println!("--- serial order: {lines:?}");
    let (tests, last) = lines.split_at(lines.len() - 1);
    assert!(
        last[0].starts_with("test result:"),
        "summary line is printed last"
    );
    let mut sorted = tests.to_vec();
    sorted.sort_unstable();
    assert_eq!(tests, sorted.as_slice(), "test lines are alphabetical");
}


The default (neither flag) **captures** passing tests' prints and throws them away
— the quiet receipt from §5.1 shows no `side effect` line at all. `--nocapture`
interleaves prints with the result lines (readable, order no longer guaranteed
across tests); `--show-output` keeps the default report clean and appends a
`successes:` archive of every test's captured output — the shape to use in CI
logs. And `--test-threads=1` pins the report to a deterministic alphabetical
order, receipt to receipt — the flag to reach for whenever a test writes to a
shared file or port (§6).


## 6. Fixtures: Setup and Teardown

unittest's `setUp`/`tearDown` run before and after *every* test method. Rust has
no hook to override — instead the idiom is honest construction: a **fixture type
with a constructor** (`new` prints the `[setup]` line) and a **`Drop` impl**
(chapter 14) printing the teardown. Because each test is a fresh function call,
each test builds its own fixture — there is no shared mutable state to trip over,
and if a test needs the *same* starting data twice, a small `fn fixture_row() ->
String` constructor (§5.1's suite has one) is the whole pattern.

The subtle receipt is the second test: `Drop` runs during **unwinding**, so
teardown executes even when the assert fails — the captured output of a failing
test shows its own `[teardown]` line:


In [ ]:
// 6.1 Fixtures as new + Drop: teardown survives a failing assert.
const FIXTURE_SRC: &str = r#"struct Fixture {
    name: &'static str,
}

impl Fixture {
    fn new(name: &'static str) -> Fixture {
        println!("[setup] {name}");
        Fixture { name }
    }
}

impl Drop for Fixture {
    fn drop(&mut self) {
        println!("[teardown] {}", self.name);
    }
}

#[test]
fn fresh_state_each_test() {
    let f = Fixture::new("fresh_state");
    assert_eq!(f.name, "fresh_state");
}

#[test]
fn teardown_runs_on_panic() {
    let f = Fixture::new("on_panic");
    assert_eq!(f.name.len(), 99, "deliberate failure");
}
"#;

let (out, code) = desk_run("fixture", FIXTURE_SRC, &["--show-output"]);
println!("{out}");
assert!(out.contains("---- fresh_state_each_test stdout ----"));
assert!(out.contains("[setup] fresh_state"));
assert!(out.contains("[teardown] fresh_state"));
let start = out.find("---- teardown_runs_on_panic stdout ----").unwrap();
let block = out[start..].to_string();
assert!(block.contains("[setup] on_panic"));
assert!(block.contains("[teardown] on_panic"));
assert!(block.contains("deliberate failure"));
assert_eq!(code, 101);


What the receipt proves: `fresh_state` runs setup → assert → teardown, all inside
the `successes:` archive; `on_panic`'s block holds **setup, the failure text, and
the teardown, in that order** — unwinding dropped the fixture on the way out. That
is the `tearDown` guarantee, and it is why the pattern needs no framework.

The same mechanics, run by hand in the kernel — a scope is the test boundary
(chapter 14's LIFO receipt, now with a reason):


In [ ]:
// 6.2 The kernel-side mirror: construction order is LIFO, the scope is the test.
struct Cursor {
    tag: &'static str,
}

impl Cursor {
    fn open(tag: &'static str) -> Cursor {
        println!("[open] {tag}");
        Cursor { tag }
    }
}

impl Drop for Cursor {
    fn drop(&mut self) {
        println!("[close] {}", self.tag);
    }
}

{
    let _a = Cursor::open("a");
    let _b = Cursor::open("b");
    println!("[work] both cursors live");
}
println!("[done] scope ended");


## 7. Organizing Tests

unittest groups tests by *class*; Rust groups them by **module** — and the
convention every Rust project ships is one line long:

```rust
#[cfg(test)]
mod tests {
    use super::*;   // pull the code under test into scope
    ...
}
```

`use super::*` reaches the *parent* module — the code under test (chapter 13's
privacy rules, flowing down). Wrapping the module in `#[cfg(test)]` is the
compile-time half of the story: the module exists **only in test builds**. The
suite's `scan` module showed the naming half; here is the full idiom and what the
compiler does to it in a production build:


In [ ]:
// 7.1 The mod tests idiom, on the desk: full-path names, cfg(test) helpers.
const MOD_TESTS_SRC: &str = r#"fn parse_fill(s: &str) -> Result<(String, i64), String> {
    let parts: Vec<&str> = s.split(',').collect();
    if parts.len() != 2 {
        return Err(format!("expected 2 fields, got {}", parts.len()));
    }
    let sym = parts[0].trim().to_uppercase();
    let qty: i64 = parts[1]
        .trim()
        .parse()
        .map_err(|_| format!("bad qty: {}", parts[1]))?;
    Ok((sym, qty))
}

#[cfg(test)]
fn fixture_row() -> String {
    "AAPL, 100".to_string()
}

#[cfg(test)]
mod tests {
    use super::*;

    #[test]
    fn parses_a_row() {
        let (sym, qty) = parse_fill("AAPL, 100").unwrap();
        assert_eq!((sym, qty), ("AAPL".to_string(), 100));
    }

    #[test]
    fn uses_the_fixture_helper() {
        let row = fixture_row();
        assert!(parse_fill(&row).is_ok());
    }
}
"#;

let (out, code) = desk_run("modtests", MOD_TESTS_SRC, &[]);
println!("{out}");
assert_eq!(code, 0);
assert!(out.contains("test tests::parses_a_row ... ok"));
assert!(out.contains("test tests::uses_the_fixture_helper ... ok"));


Now flip the build to **production** (`rustc` without `--test`, `cfg(test)` off):
the whole module vanishes, `fixture_row` with it — and the compiler is *silent*
about the `#[test]` functions it stripped, no dead-code warnings, because they are
`cfg(test)`-gated just like the module. What the stripping does *not* forgive is
being **named**: a call to a stripped function is `E0425`, and the production
helper used only by tests now draws a `dead_code` warning of its own. Both
receipts, captured as data:


In [ ]:
// 7.2 The production build strips tests silently -- until you name one.
const PROD_OK: &str = r#"fn parse_fill(s: &str) -> Result<(String, i64), String> {
    let parts: Vec<&str> = s.split(',').collect();
    Ok((parts[0].to_uppercase(), 0))
}

#[cfg(test)]
fn fixture_row() -> String {
    "AAPL, 100".to_string()
}

#[test]
fn parses_ok() {
    let (sym, _) = parse_fill("AAPL, 100").unwrap();
    assert_eq!(sym, "AAPL");
}

fn main() {
    println!("production build: no tests here");
}
"#;

let stderr = desk_stderr("prodok", PROD_OK, false);
println!("--- plain build stderr:\n{stderr}");
assert!(stderr.contains("warning: function `parse_fill` is never used"));
assert!(stderr.contains("dead_code"));

const PROD_NAMED: &str = r#"fn parse_fill(s: &str) -> Result<(String, i64), String> {
    Ok(("AAPL".to_string(), 100))
}

#[test]
fn parses_ok() {
    assert!(parse_fill("AAPL, 100").is_ok());
}

fn main() {
    parse_fill("AAPL, 100").unwrap();
    parses_ok();
}
"#;

let stderr = desk_stderr("prodnamed", PROD_NAMED, false);
println!("--- naming a stripped test:\n{stderr}");
assert!(stderr.contains("error[E0425]: cannot find function `parses_ok`"));


In [ ]:
// 7.3 The kernel is a production build: cfg!(test) is false, test fns are gone.
println!("cfg!(test) in this kernel = {}", cfg!(test));
assert!(!cfg!(test));


The probe receipt behind §7.3, worth restating exactly: a kernel cell defined
`#[test] fn k_ok()` and the **next cell's** `k_ok();` failed with `E0425 cannot
find function` — the attribute stripped the function from its own cell, before
persistence ever saw it. (`#[cfg(test)] mod` strips the same way, as `E0433`.)
That is why this notebook's `#[test]` only ever appears *inside desk strings*.

One more organization boundary, named so you can look it up: Rust also has
**integration tests** — a `tests/` directory in a cargo project, one binary per
file, each compiled against the crate's *public* API only. Same harness, same
output, no `#[cfg(test)]` needed; it is cargo-side layout, out of scope for a
notebook, and §9's mapping table marks the spot.


## 8. Exit Codes and the Result Line

`cargo test` is an exit code and a grammar:

- the binary exits **`0`** if every selected test passed, **`101`** if any failed —
  the number a CI system actually reads (§1.3 first printed it);
- the last line, `test result: ok. 5 passed; 0 failed; 1 ignored; 0 measured;
  0 filtered out; finished in 0.00s`, is a **machine-readable summary** — five
  counters and a wall time.

`measured` counts benchmarks (always `0` outside `cargo bench`), and `filtered
out` is the selection bookkeeping from §5. Since chapter 24 taught
string-into-structure, the summary line might as well *become* structure:


In [ ]:
// 8.1 Parse the result line into a HashMap<String, u32> -- the CI's job as data.
use std::collections::HashMap;

let (out, code) = desk_run("suite", SUITE_SRC, &[]);
let line = out.lines()
    .find(|l| l.starts_with("test result:"))
    .map(|l| l.to_string())
    .unwrap();
println!("line: {line}");
assert_eq!(code, 0);

let body = line.split_once(". ").unwrap().1.to_string();
let mut counts: HashMap<String, u32> = HashMap::new();
for part in body.split(';') {
    let part = part.trim();
    if part.starts_with("finished") {
        continue;
    }
    let (n, label) = part.split_once(' ').unwrap();
    counts.insert(label.to_string(), n.parse().unwrap());
}
println!("parsed: {counts:?}");
assert_eq!(counts["passed"], 5);
assert_eq!(counts["failed"], 0);
assert_eq!(counts["ignored"], 1);
assert_eq!(counts["measured"], 0);
assert_eq!(counts["filtered out"], 0);


In [ ]:
// 8.2 The verdict line follows the counts: "ok." only when failed == 0 --
// and the zero-match quirk from section 5: nothing selected still exits 0.
let (out, code) = desk_run("broken", FIRST_BROKEN, &[]);
let line = out.lines()
    .find(|l| l.starts_with("test result:"))
    .map(|l| l.to_string())
    .unwrap();
println!("failing suite: {line}");
assert!(line.contains("FAILED. 1 passed; 1 failed"));
assert_eq!(code, 101);

let (none, code) = desk_run("suite", SUITE_SRC, &["nope", "--exact"]);
assert!(none.contains("test result: ok. 0 passed"));
assert_eq!(code, 0);
println!("zero-match filter: exit {code} -- 'ran nothing' is not a failure");


## 9. The unittest Mapping

The whole dictionary on one table. The Rust column names the section that proved
each row.

| Python `unittest` | Rust (this chapter) |
|---|---|
| `class TestFill(unittest.TestCase)` | `#[cfg(test)] mod tests { use super::*; }` (§7) |
| `def test_parses_ok(self)` | `#[test] fn parses_ok()` — no args, no self (§1) |
| `self.assertEqual(a, b)` | `assert_eq!(a, b)` (§2) |
| `self.assertNotEqual(a, b)` | `assert_ne!(a, b)` (§2) |
| `self.assertTrue(x)` / `assertFalse(x)` | `assert!(x)` / `assert!(!x)` (§2) |
| `self.assertIn(k, d)` | `assert!(d.contains(&k))` — iterator/slice methods, not a macro (§2) |
| `self.assertRaises(ValueError)` | `#[should_panic(expected = "...")]` (§3) |
| a test that just returns | `-> Result<(), E>`: `Err` counts as a failure (§4) |
| `@unittest.skip("reason")` | `#[ignore = "reason"]` (§5) |
| `-k pattern` | a bare name-filter argument; `--exact` for whole paths (§5) |
| `setUp` / `tearDown` | a `new()` constructor + `Drop` impl, per test (§6) |
| `self.subTest(...)` | separate `#[test]` fns — one line and one capture block each (§2) |
| captured stdout / `-v` | `--show-output` (archive) / `--nocapture` (inline) (§5) |
| `python -m unittest` | `cargo test` — or the raw binary `rustc --test` builds (§0, §8) |

Two rows have **no unittest twin** and are worth calling out: the **exit-code
contract** (`0`/`101`, §8 — CI reads it, not a human), and **`Termination`** — the
trait that decides which return types a test (or `main`) may have (§10). Rust's
runner also reports `filtered out` counts and `- should panic` annotations inline;
unittest's runner is quieter about selection.


## 10. E-Codes, Traps, and Kernel Facts

The harness validates test **signatures** at compile time — three refusals every
testing chapter should show, captured through `desk_stderr` as data:


In [ ]:
// 10.1 The harness rejects test fns with arguments -- and non-Termination returns.
const ARG_TEST: &str = r#"#[test]
fn with_arg(x: i32) {
    let _ = x;
}

fn main() {}
"#;

let stderr = desk_stderr("argtest", ARG_TEST, true);
println!("{stderr}");
assert!(stderr.contains("functions used as tests can not have any arguments"));

const RETURN_TEST: &str = r#"#[test]
fn returns_i32() -> i32 {
    4
}

fn main() {}
"#;

let stderr = desk_stderr("rettest", RETURN_TEST, true);
println!("{stderr}");
assert!(stderr.contains("error[E0277]"));
assert!(stderr.contains("`i32: Termination` is not satisfied"));
assert!(stderr.contains("in this attribute macro expansion"));


In [ ]:
// 10.2 Attributes only pair with #[test]: #[ignore] alone is a dead function.
const IGNORE_ALONE: &str = r#"#[test]
fn real_test() {
    assert_eq!(1 + 1, 2);
}

#[ignore]
fn ignored_but_not_a_test() {
    assert_eq!(1, 1);
}

fn main() {}
"#;

let stderr = desk_stderr("igalone", IGNORE_ALONE, true);
println!("--- compile stderr:\n{stderr}");
assert!(stderr.contains("warning: function `ignored_but_not_a_test` is never used"));

// desk_run asserts a *clean* compile, which this source fails on purpose
// (the dead-code warning IS the receipt), so run the compiled exe directly:
let run = Command::new("./scratch25/desk_igalone.exe").output().unwrap();
let out = String::from_utf8_lossy(&run.stdout).into_owned();
let code = run.status.code().unwrap_or(-1);
println!("--- run:\n{out}");
assert!(out.contains("running 1 test"));
assert!(out.contains("test result: ok. 1 passed; 0 failed; 0 ignored"));
assert_eq!(code, 0);


The run says it all: `0 ignored` — `#[ignore]` without `#[test]` marks nothing,
the function never ran, and the compiler waved a `dead_code` warning. The
companion probe fact: `#[should_panic]` *without* `#[test]` compiles as a plain
function (the kernel probe called one and got the panic, uncaught). Attributes in
Rust are inert decorations; only the combination `#[test]`-plus-harness gives them
meaning.

| Diagnostic | The story it tells |
|---|---|
| `E0425` cannot find function | naming a **stripped** `#[test]` fn — in a plain build *or* from a later kernel cell (§7.2, §7.3) |
| `E0433` cannot find module or crate | naming a `#[cfg(test)]` `mod` in a non-test build (§7.2's cousin; probed in the kernel too) |
| `E0277` `T: Termination` | a `#[test]` fn returning something unharnessable — `i32` instead of `()` or `Result` (§10.1) |
| `E0277` `T: Debug` (via `unwrap_err`) | `unwrap_err` must print the `Ok` value it discards; give the struct a `Debug` derive (§4, probe-caught) |
| `functions used as tests can not have any arguments` | harness signature check — tests take nothing (§10.1) |
| `warning: function ... is never used` | a helper used only by tests in a plain build (§7.2) — or `#[ignore]` without `#[test]` (§10.2) |
| `panic did not contain expected string` | `#[should_panic(expected)]` mismatch: real message vs expected substring, printed as a pair (§3.2) |
| exit `101` | any selected test failed; `0` = green, including the zero-match quirk (§8) |

Kernel facts, one line each: `#[test]` fns are **stripped before persistence**
(E0425 from the very next cell); `cfg!(test)` is `false`; the desk recompiles its
source on every call (~0.9 s) and the harness itself runs in ~10 ms; and no cell
here ever panics on purpose — every failure you saw arrived **captured**, through
`Command`'s stdout, exactly as chapter 23 promised subprocesses would.


## 11. Mental Model: The Receipts

One suite, every shape from the chapter, one run — a failing test on purpose so
the *whole* receipt lands as data. Read it top to bottom before moving on: this is
what a mixed report looks like when a colleague hands it to you.


In [ ]:
// 11.1 The full matrix: pass, assert-failure, ignore, should_panic, Err-failure.
const MATRIX_SRC: &str = r#"fn parse_fill(s: &str) -> Result<(String, i64), String> {
    if s.trim().is_empty() {
        return Err("empty input".to_string());
    }
    let parts: Vec<&str> = s.split(',').collect();
    let qty: i64 = parts
        .get(1)
        .ok_or("missing qty field")?
        .trim()
        .parse()
        .map_err(|e| format!("bad qty: {e}"))?;
    Ok((parts[0].trim().to_uppercase(), qty))
}

#[test]
fn parses_ok() {
    let (sym, qty) = parse_fill("AAPL, 100").unwrap();
    assert_eq!((sym, qty), ("AAPL".to_string(), 100));
}

#[test]
fn fails_on_assert() {
    let (_, qty) = parse_fill("MSFT, 7").unwrap();
    assert_eq!(qty, 0, "MSFT should not trade today");
}

#[test]
#[ignore = "would need the real feed"]
fn live_feed_smoke() {
    parse_fill("AAPL, 1").unwrap();
}

#[test]
#[should_panic(expected = "empty input")]
fn panics_on_empty() {
    parse_fill("").unwrap();
}

#[test]
fn err_is_failure() -> Result<(), String> {
    parse_fill("AAPL, lots")?;
    Ok(())
}
"#;

let (out, code) = desk_run("matrix", MATRIX_SRC, &["--show-output"]);
println!("{out}");
println!("exit code: {code}");
assert!(out.contains("test parses_ok ... ok"));
assert!(out.contains("test fails_on_assert ... FAILED"));
assert!(out.contains("test live_feed_smoke ... ignored, would need the real feed"));
assert!(out.contains("test panics_on_empty - should panic ... ok"));
assert!(out.contains("test err_is_failure ... FAILED"));
assert!(out.contains("Error: \"bad qty: invalid digit found in string\""));
assert!(out.contains("test result: FAILED. 2 passed; 2 failed; 1 ignored"));
assert_eq!(code, 101);


Count the story the receipt tells: `2 passed; 2 failed; 1 ignored` — the failing
assert printed its `left`/`right` under `fails_on_assert`, the `?`-propagated
`Err` printed `Error: "bad qty: invalid digit found in string"` under `err_is_failure`,
`panics_on_empty` **passed because it panicked with the right substring**, and the
ignored test's reason explains its own absence. Two failures, two *different*
failure channels, both reaching you through the same capture-block grammar.


## 12. Cheat Sheet

```rust
// --- writing tests ------------------------------------------------------
#[test]
fn name_meaningful() { /* arrange, act, assert */ }

#[test]
fn returns_result() -> Result<(), String> { /* ...? */ Ok(()) }  // Err = failure

#[test]
#[should_panic(expected = "substring of the message")]
fn rejects_bad() { /* must panic, with that substring */ }

#[test]
#[ignore = "why it is parked"]
fn expensive() { /* runs only with --ignored */ }

// --- the assert family --------------------------------------------------
assert!(cond);                        // assertTrue;  no message: prints the expr
assert!(cond, "msg {}", x);           // custom payload, format! syntax
assert_eq!(a, b);  assert_ne!(a, b);  // prints left/right, Debug-formatted
assert!(matches!(v, Ok(_)));          // shape checks (ch08's idiom)

// --- organizing ---------------------------------------------------------
#[cfg(test)]
mod tests {
    use super::*;                     // the code under test
    #[test]
    fn full_path_is_the_name() {}
}
// fixtures: fn new() -> Fixture (setup) + impl Drop (teardown), one per test

// --- the harness --------------------------------------------------------
// rustc --test file.rs   build the test binary (libtest = cargo test's runner)
// ./file.exe             run every test          -> exit 0 | 101
// ./file.exe parse       substring filter on the full test path
// ./file.exe mod::t --exact            one whole-path test
// --list  --ignored  --nocapture  --show-output  --test-threads=1
// "test result: ok. 5 passed; 0 failed; 1 ignored; 0 measured; 0 filtered out"
```

The kernel caveat, one line: `#[test]` never survives a kernel cell — tests live
in files, and a `rustc --test` build is the notebook-shaped way to run them.


## 13. Review and What Comes Next

**Why is `#[test]` stripped in this kernel, and what is the desk doing about it?**
The attribute gates on `cfg(test)`; the kernel builds without the harness, so
`#[test]` functions (and `#[cfg(test)]` modules) vanish — the desk re-creates a
*harness build* with `rustc --test` and runs its binary as a child process,
capturing stdout and the exit code as data (§0, §7.3).

**What makes a test a test, beyond the attribute?** The signature contract: no
arguments, `()` or a `Termination`-implementing return (`Result<(), E>` with
`E: Debug`), and the body either returning quietly, returning `Err`, or panicking
(§10.1, §4, §3).

**How does `Err` differ from a panic as a failure?** Both fail the test; `Err` is
printed as data (`Error: "..."`) without unwinding, `?` works naturally in the
body, and `#[should_panic]` is the *expectation* side of panicking — substring
matched, `panic did not contain expected string` on mismatch (§3, §4).

**What does the exit code mean to a machine?** `0` green, `101` red, and the
`test result:` line carries five counters — `passed / failed / ignored / measured
/ filtered out` — that CI parses; a zero-match filter still exits `0` (§8).

**Where do `setUp`/`tearDown` go?** Into a fixture type: the constructor is
`setUp`, `Drop` is `tearDown`, each test builds its own, and unwinding guarantees
the teardown even on failure (§6).

**What Comes Next.** Chapter 26 closes the series with **macros** — and this
chapter quietly used two: `#[test]` and `#[should_panic]` are *attribute* macros
(E0277 even said so: `in this attribute macro expansion`), while `assert_eq!` and
`matches!` are `macro_rules!` declarations. The machinery that lets a test take no
arguments, or an assert capture its own expression text, is the last idea on the
list.


In [ ]:
// 13.1 Desk teardown: the suite has been run, the directory is disposable.
use std::path::Path;

let existed = Path::new("scratch25").exists();
if existed {
    fs::remove_dir_all("scratch25").unwrap();
}
println!(
    "scratch25 existed: {existed} -- removed: {}",
    !Path::new("scratch25").exists()
);
